<h1>Chapter 4 - Preparing Data for Vector Stores</h1>
<i>Designing data preparation pipelines to preprocess and chunk text data.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch03_loading_data/loading_data_to_RAG.ipynb)

---

This notebook is for Chapter 4 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [ ]:
!pip install PyPDF2==3.0.1
!pip install pandas==2.2.3
!pip install pydantic==2.11.5
!pip install openai==1.83.0
!pip install matplotlib==3.10.3
!pip install scikit-learn==1.6.1
!pip install python-docx==1.1.2
!pip install nltk==3.9.1
!pip install langchain==0.3.25
# !pip install langchain_openai==0.3.21
!pip install langchain-experimental==0.3.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.5 MB/s eta 0:00:00


### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
!git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
%cd RAG-with-Python-Cookbook
!git sparse-checkout init --cone
!git sparse-checkout set datasets
!git checkout
!cp -r datasets /content/datasets


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

1. Adding Metadata to Enable Metadata Filtering

In [ ]:
# tag::load_pdf_and_metadata[]
import PyPDF2
import os

file_path = "../datasets/pdf_files/attention_is_all_you_need_paper.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    metadata = reader.metadata

    text = ""
    for page in reader.pages:
        text += page.extract_text()
# end::load_pdf_and_metadata[]

In [ ]:
metadata

In [ ]:
# tag::generate_customized_metadata[]
metadata_ext = dict(metadata)
metadata_ext["page_count"] = len(reader.pages)
metadata_ext["file_size"] = os.path.getsize(file_path)
metadata_ext["file_name"] = os.path.basename(file_path)
metadata_ext["file_path"] = file_path
metadata_ext["text_length"] = len(text)
# end::generate_customized_metadata[]

In [ ]:
metadata_ext

In [ ]:
# tag::extract_metadata_from_text_using_LLMs[]
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class AuthorContact(BaseModel):
    name: str
    company: str
    email: list[str]

class Contacts(BaseModel):
    entries: list[AuthorContact]

system_message = """Extract the contact information of all authors."""

response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": text,
        },
    ],
    text_format=Contacts,
)

author_contacts = response.output_parsed

metadata_ext["author_contacts"] = author_contacts
# end::extract_metadata_from_text_using_LLMs[]



In [ ]:
metadata

In [ ]:
metadata

### 2.2 Enhancing Data Quality by Replacing Abbreviations and Technical Terms

In [ ]:
# tag::load_file_and_replace_abbreviations[]
import re

abbreviations_dict = {
    "NLP": "Natural Language Processing",
    "RNN": "Recurrent Neural Network",
    "LSTM": "Long Short-Term Memory",
    "GRU": "Gated Recurrent Unit",
    "TF": "Transformer",
    "MHA": "Multi-Head Attention",
    "FFN": "Feed-Forward Network",
}

# Load the sample text file
file_path = "../datasets/text_files/blog_post_transformers.txt"
with open(file_path, "r") as file:
    text = file.read()

# Replace abbreviations in the text
for abbr, full_form in abbreviations_dict.items():
    text = re.sub(rf"\b{abbr}\b", f"{full_form} ({abbr})", text)
# end::load_file_and_replace_abbreviations[]


In [ ]:
text

In [ ]:
# tag::make_text_chunks_self_explanatory[]
import os
from openai import OpenAI

file_path = "../datasets/text_files/EMEA_drives_revenue.txt"

with open(file_path, "r") as file:
    text = file.read()

prompt = f"""
    The text below contains a financial report including a lot of abbreviations and
    technical terms from the finance domain. Please replace the abbreviations with
    their full forms and provide a brief explanation of the technical terms, so the
    whole text get's easier to read and understandable for everyone.

    Make sure it's easy enough, that a 10 years old school kid could understand it.

    Often used abbreviations are:
    - EMEA: Europe, Middle East, and Africa
    - BD: Business Development
    - YoY: Year-over-Year
    - APAC: Asia-Pacific

    Text:
    {text}
    """.strip()

client = OpenAI()
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-4o",
)

enhanced_text = chat_completion.choices[0].message.content

# end::make_text_chunks_self_explanatory[]

In [ ]:
enhanced_text

In [ ]:
# write enhanced_text to a new .txt file
output_file_path = "../datasets/text_files/EMEA_drives_revenue_enhanced.txt"
with open(output_file_path, "w") as file:
    file.write(enhanced_text)

### 2.3 Improving Search Accuracy by Embedding Hypothetical Questions

In [ ]:
import PyPDF2

file_path = "../datasets/pdf_files/AI_in_Factories_Discussion_Cleaned.pdf"

with open(file_path, "rb") as file:
    # Create PDF reader object
    reader = PyPDF2.PdfReader(file)

    # Extract text from all pages
    text = ""
    for page in reader.pages:
        text += page.extract_text()


In [ ]:
text

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
    ],
)

text_chunks = text_splitter.split_text(text)

In [ ]:
# tag::generate_hypothetical_questions[]
import textwrap
from openai import OpenAI
from pydantic import BaseModel

file_path = "../datasets/text_files/AI_in_factories_chat.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

client = OpenAI()

prompt = textwrap.dedent(
    f"""
    Below you can find a chat history between two students.

    Please generate 5 hypothetical questions that could be
    answered using the information from the discussion.
    The questions should focus on key details, definitions, and
    information present in the text.

    Chat History:
    {text}
    """
)

class HypotheticalQuestions(BaseModel):
    questions: list[str]

result = client.responses.parse(
    model="gpt-4o",
    input=prompt,
    text_format=HypotheticalQuestions,
)

hypothetical_questions = result.output_parsed.questions
hypothetical_questions
# end::generate_hypothetical_questions[]


In [ ]:
hypothetical_questions

### 2.4 Splitting Documents Using Character Splitting

In [ ]:
# tag::character_text_splitting[]
from langchain.text_splitter import CharacterTextSplitter

file_path = "../datasets/text_files/blog_post_transformers.txt"

# Load example document
with open(file_path, "r") as file:
    text = file.read()

text_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separator="",
    length_function=len,
)

text_chunks = text_splitter.create_documents([text])
# end::character_text_splitting[]

In [ ]:
text_chunks

### 2.5 Splitting Documents Using Recursive Text Splitters

In [ ]:

# tag::recursive_chunking[]
from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2

file_path = "../datasets/pdf_files/daily_insights.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_text(text)
# end::recursive_chunking[]


In [ ]:
chunks

### 2.6 Document Aware Splitting

In [ ]:

# tag::document_aware_text_splitter[]
import os

file_path = "../datasets/markdown_files/random_md_code.md"
file_extension = os.path.splitext(file_path)[1]

with open(file_path, "r") as file:
    file_text = file.read()

from langchain_text_splitters import (
    PythonCodeTextSplitter,
    LatexTextSplitter,
    MarkdownHeaderTextSplitter,
)

if file_extension == ".py":
    splitter = PythonCodeTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".tex":
    splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".md":
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

chunks = splitter.split_text(file_text)
# end::document_aware_text_splitter[]

In [ ]:
chunks

### 2.7 Splitting the Text Using Semantic Aware Chunkers

In [ ]:
from docx import Document
from openai import OpenAI

file_path = "../datasets/text_files/random-text-about-5-different-stories.txt"

# read the text from the file
with open(file_path, "r") as file:
    text = file.read()

In [ ]:
# tag::langchain_semantic_text_splitting[]
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

chunks = text_splitter.split_text(text)
# end::langchain_semantic_text_splitting[]

In [ ]:
# chunks

In [ ]:
# import pandas as pd

# # initialize the api key
# client = OpenAI()
# embedding_model = "text-embedding-3-small"

# import os

# # Initialize the OpenAI client with your API key
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# # create new data frame using text chunks list
# embeddings_df = pd.DataFrame(text_chunks).rename(columns={0: "text_chunk"})

# # helper function to get the embeddings for a text chunk
# def _get_embeddings(text_chunk, client, embedding_model):
#     embedding = (
#         client.embeddings.create(input=[text_chunk], model=embedding_model)
#         .data[0]
#         .embedding
#     )

#     return embedding

# # iterate over embeddings_df["text_chunks"] and create a new data frame with the embeddings
# embeddings_df["embeddings"] = embeddings_df["text_chunk"].apply(
#     _get_embeddings, client=client, embedding_model="text-embedding-3-small"
# )

# # split the embeddings column into individual columns for each vector dimension
# embeddings_df = embeddings_df["embeddings"].apply(pd.Series)
# embeddings_df["text_chunk"] = text_chunks

# embeddings_df = from_text_to_embeddings(
#     chunks, client, embedding_model
# )

### 2.8 Splitting Text Using Agentic Chunkers

In [ ]:
# tag::agentic_chunking_create_propositions[]
from langchain import hub
from langchain_openai import ChatOpenAI  # Import ChatOpenAI
from pydantic import BaseModel
from typing import List

# pull the prompt template from the langchain hub
obj = hub.pull("wfh/proposal-indexing")

llm = ChatOpenAI(model="gpt-4o")

class Sentences(BaseModel):
    sentences: List[str]

extraction_llm = llm.with_structured_output(Sentences)

# Create the sentence extraction chain
extraction_chain = obj | extraction_llm

propositions = extraction_chain.invoke(
    """
    On July 20, 1969, astronaut Neil Armstrong walked on the moon .
    He was leading the NASA's Apollo 11 mission.
    Armstrong famously said, "That's one small step for man, one
    giant leap for mankind" as he stepped onto the lunar surface.
    """
)

print(propositions)
# end::agentic_chunking_create_propositions[]

In [ ]:
for sentence in propositions.sentences:
    print(sentence)